In [0]:
%run "/Workspace/Users/tarunvinodh@gmail.com/Notebook-1 ADLS Connection"

In [0]:
from pyspark.sql.functions import *

In [0]:
address_schema = StructType([
    StructField("AddressSK", IntegerType(), True),
    StructField("CustomerID", IntegerType(), True),
    StructField("City", StringType(), True),
])
customers_schema = StructType([
    StructField("CustomerID", IntegerType(), False),
    StructField("FirstName", StringType(), True),
    StructField("Email", StringType(), True)
])
loyalty_schema = StructType([
    StructField("LoyaltyID", IntegerType(), False),
    StructField("CustomerID", IntegerType(), False),
    StructField("PointsBalance", IntegerType(), True),
    StructField("MemberTier", StringType(), True)
])
orders_schema = StructType([
    StructField("OrderID", IntegerType(), False),
    StructField("CustomerID", IntegerType(), False),
    StructField("OrderDate", TimestampType(), True),
    StructField("TotalAmount", DoubleType(), True)
])
tickets_schema = StructType([
    StructField("TicketID", IntegerType(), False),
    StructField("CustomerID", IntegerType(), False),
    StructField("IssueType", StringType(), True),
    StructField("Status", StringType(), True)
])


In [0]:
df_address=spark.read.format("csv").schema(address_schema).option("header", "true").load(f"{adlspath}/Addresses/Addresses.csv")
df_customers=spark.read.format("csv").schema(customers_schema).option("header", "true").load(f"{adlspath}/Customers/Customers")
df_loyalty=spark.read.format("csv").schema(loyalty_schema).option("header","true").load(f"{adlspath}/LoyaltyPoints/LoyaltyPoints")
df_orders=spark.read.format("csv").schema(orders_schema).option("header", "true").load(f"{adlspath}/Orders/Orders")
df_tickets=spark.read.format("csv").schema(tickets_schema).option("header", "true").load(f"{adlspath}/SupportTickets/SupportTickets")


###  NULL CHECK

In [0]:
df_address=df_address.filter(df_address.AddressSK.isNotNull())
df_customers=df_customers.filter(df_customers.CustomerID.isNotNull())
df_loyalty=df_loyalty.filter(df_loyalty.LoyaltyID.isNotNull())
df_orders=df_orders.filter(df_orders.OrderID.isNotNull())
df_tickets=df_tickets.filter(df_tickets.TicketID.isNotNull())


In [0]:
df_customers.display()

CustomerID,FirstName,Email
101,Alice,alice.j@email.com
102,Bob,bsmith99@email.com
103,Charlie,charlie.d@email.com
104,Diana,diana.p@email.com
105,Ethan,hunt.e@email.com
105,Ethan,hunt.e@email.com


### Create New Columns

In [0]:
df_address=df_address.withColumn("Country", when(col("City") == "New York", "USA").when(col("City") == "Manchester", "UK").when(col("City") == "Toronto", "Canada").otherwise("Germany"))

In [0]:
df_customers=df_customers.withColumn("LastName",when(col("FirstName") == "Alice", "Smith").when(col("FirstName") == "Bob", "Jones").when(col("FirstName") == "Charlie", "Williams").otherwise("Unknown"))

In [0]:
df_customers.display()

CustomerID,FirstName,Email,LastName
101,Alice,alice.j@email.com,Smith
102,Bob,bsmith99@email.com,Jones
103,Charlie,charlie.d@email.com,Williams
104,Diana,diana.p@email.com,Unknown
105,Ethan,hunt.e@email.com,Unknown
105,Ethan,hunt.e@email.com,Unknown


In [0]:
df_customers=df_customers.replace("Unknown","ABCD","LastName")
df_customers.display()

CustomerID,FirstName,Email,LastName
101,Alice,alice.j@email.com,Smith
102,Bob,bsmith99@email.com,Jones
103,Charlie,charlie.d@email.com,Williams
104,Diana,diana.p@email.com,ABCD
105,Ethan,hunt.e@email.com,ABCD
105,Ethan,hunt.e@email.com,ABCD


### Replace Null Values

In [0]:
df_loyalty.display()

LoyaltyID,CustomerID,PointsBalance,MemberTier
1,101,1250,Gold
2,102,450,Silver
3,103,800,Silver
4,104,2100,"""Platinum""å"
5,105,null,null
6,105,50,Bronze


In [0]:
df_loyalty=df_loyalty.fillna({
    "MemberTier": "N/A",
    "PointsBalance": -1
})
df_loyalty.display()

LoyaltyID,CustomerID,PointsBalance,MemberTier
1,101,1250,Gold
2,102,450,Silver
3,103,800,Silver
4,104,2100,"""Platinum""å"
5,105,-1,N/A
6,105,50,Bronze


### DUPLICATES CHECK

In [0]:
display(df_address); display(df_loyalty); display(df_orders); display(df_tickets)

AddressSK,CustomerID,City,Country
1,101,New York,USA
2,102,Manchester,UK
3,103,Kitchener,Germany
4,104,Berlin,Germany
5,105,Frankfurt,Germany


LoyaltyID,CustomerID,PointsBalance,MemberTier
1,101,1250,Gold
2,102,450,Silver
3,103,800,Silver
4,104,2100,"""Platinum""å"
5,105,-1,N/A
6,105,50,Bronze


OrderID,CustomerID,OrderDate,TotalAmount
5001,101,2026-01-10T00:00:00Z,150.0
5002,103,2026-01-12T00:00:00Z,45.0
5003,102,2026-01-15T00:00:00Z,300.0
5004,105,2026-02-01T00:00:00Z,12.0
5005,104,2026-02-03T00:00:00Z,89.0
5005,104,2026-02-04T00:00:00Z,89.0


TicketID,CustomerID,IssueType,Status
901,102,Refund,Closed
902,105,Login,Open
903,101,Shipping,Pending
904,104,Billing,Closed
905,103,Feedback,Open
905,103,null,Open


In [0]:
df_customers=df_customers.dropDuplicates(["CustomerID"])
df_tickets=df_tickets.dropDuplicates(["TicketID"])
df_customers.display()
df_tickets.display()

CustomerID,FirstName,Email,LastName
101,Alice,alice.j@email.com,Smith
102,Bob,bsmith99@email.com,Jones
103,Charlie,charlie.d@email.com,Williams
104,Diana,diana.p@email.com,ABCD
105,Ethan,hunt.e@email.com,ABCD


TicketID,CustomerID,IssueType,Status
901,102,Refund,Closed
902,105,Login,Open
903,101,Shipping,Pending
904,104,Billing,Closed
905,103,Feedback,Open


In [0]:
from pyspark.sql import Window

In [0]:
w = Window.partitionBy("CustomerID").orderBy(
    when((col("PointsBalance") == -1) | (col("MemberTier") == "N/A"), 1).otherwise(0)
)

In [0]:
df_loyalty_with_rn = df_loyalty.withColumn("row_num", row_number().over(w))

In [0]:
df_loyalty=df_loyalty_with_rn.filter(col("row_num") == 1).drop("row_num")
df_loyalty.display()

LoyaltyID,CustomerID,PointsBalance,MemberTier
1,101,1250,Gold
2,102,450,Silver
3,103,800,Silver
4,104,2100,"""Platinum""å"
6,105,50,Bronze


In [0]:
from pyspark.sql.functions import regexp_replace, col

df_loyalty=df_loyalty.withColumn("MemberTier",regexp_replace(col("MemberTier"), r'[^a-zA-Z]', ''))



In [0]:
df_loyalty.display()

LoyaltyID,CustomerID,PointsBalance,MemberTier
1,101,1250,Gold
2,102,450,Silver
3,103,800,Silver
4,104,2100,Platinum
6,105,50,Bronze


In [0]:
df_orders.display()

OrderID,CustomerID,OrderDate,TotalAmount
5001,101,2026-01-10T00:00:00Z,150.0
5002,103,2026-01-12T00:00:00Z,45.0
5003,102,2026-01-15T00:00:00Z,300.0
5004,105,2026-02-01T00:00:00Z,12.0
5005,104,2026-02-03T00:00:00Z,89.0
5005,104,2026-02-04T00:00:00Z,89.0


In [0]:
w1 = Window.partitionBy("CustomerID").orderBy(col("OrderDate").desc())

In [0]:
df_orders_with_rn = df_orders.withColumn("row_num", row_number().over(w1))
df_orders_with_rn.display()

OrderID,CustomerID,OrderDate,TotalAmount,row_num
5001,101,2026-01-10T00:00:00Z,150.0,1
5003,102,2026-01-15T00:00:00Z,300.0,1
5002,103,2026-01-12T00:00:00Z,45.0,1
5005,104,2026-02-04T00:00:00Z,89.0,1
5005,104,2026-02-03T00:00:00Z,89.0,2
5004,105,2026-02-01T00:00:00Z,12.0,1


In [0]:
df_orders=df_orders_with_rn.filter(col("row_num") == 1).drop("row_num")
df_orders.display()

OrderID,CustomerID,OrderDate,TotalAmount
5001,101,2026-01-10T00:00:00Z,150.0
5003,102,2026-01-15T00:00:00Z,300.0
5002,103,2026-01-12T00:00:00Z,45.0
5005,104,2026-02-04T00:00:00Z,89.0
5004,105,2026-02-01T00:00:00Z,12.0


In [0]:
# df_address.write.format("delta").saveAsTable("address")
# df_customers.write.format("delta").saveAsTable("customers")
# df_loyalty.write.format("delta").saveAsTable("loyalty")
# df_orders.write.format("delta").saveAsTable("orders")
# df_tickets.write.format("delta").saveAsTable("tickets")

---------------------------------------------------------------------------
AnalysisException                         Traceback (most recent call last)
File <command-7863148245605943>, line 1
----> 1 df_address.write.format("delta").saveAsTable("address")
      2 df_customers.write.format("delta").saveAsTable("customers")
      3 df_loyalty.write.format("delta").saveAsTable("loyalty")

File /databricks/spark/python/pyspark/sql/connect/readwriter.py:737, in DataFrameWriter.saveAsTable(self, name, format, mode, partitionBy, **options)
    735 self._write.table_name = name
    736 self._write.table_save_method = "save_as_table"
--> 737 _, _, ei = self._spark.client.execute_command(
    738     self._write.command(self._spark.client), self._write.observations
    739 )
    740 self._callback(ei)

File /databricks/spark/python/pyspark/sql/connect/client/core.py:1589, in SparkConnectClient.execute_command(self, command, observations, extra_request_metadata)
   1587     req.user_context.user_

In [0]:
%sql
desc formatted address

col_name,data_type,comment
AddressSK,int,null
CustomerID,int,null
City,string,null
Country,string,null
,,
# Delta Statistics Columns,,
Column Names,"AddressSK, CustomerID, City, Country",
Column Selection Method,first-32,
,,
# Detailed Table Information,,


In [0]:
%sql
select * from address

AddressSK,CustomerID,City,Country
1,101,New York,USA
2,102,Manchester,UK
3,103,Toronto,Canada
4,104,Berlin,Germany


### SCD TYPE 1 - ADDRESS

In [0]:
%sql
Create Table if not exists address_scdtype1
(
  AddressSK int,
  CustomerID int,
  City string,
  Country String,
  Hashvalue bigint,
  CreatedDate timestamp,
  UpdatedDate timestamp,
  CreatedBy string,
  UpdatedBy string
)

In [0]:
from delta.tables import DeltaTable
table_name='address_scdtype1'
delta_tgt=DeltaTable.forName(spark, table_name)

In [0]:
from pyspark.sql.functions import *
df_hash=df_address.withColumn('src_hash',crc32(concat(col('AddressSK'),col('CustomerID'),col('City'),col('Country'))))
df_hash.display()

AddressSK,CustomerID,City,Country,src_hash
1,101,New York,USA,403127107
2,102,Manchester,UK,2429636690
3,103,Kitchener,Germany,2448937089
4,104,Berlin,Germany,2681540978
5,105,Frankfurt,Germany,3153943322


In [0]:
(
delta_tgt.alias("tgt").merge(
    df_hash.alias("src"),
    "tgt.CustomerID = src.CustomerID",
).whenMatchedUpdate(condition="tgt.Hashvalue!=src.src_hash",
                    set={
                        "tgt.AddressSK":"src.AddressSK",
                        "tgt.CustomerID":"src.CustomerID",
                        "tgt.City":"src.City",
                        "tgt.Country":"src.Country",
                        "tgt.Hashvalue":"src.src_hash",
                        "tgt.UpdatedDate":current_timestamp(),
                        "tgt.UpdatedBy":lit("Databricks"),
                        "tgt.CreatedDate":current_timestamp(),
                        "tgt.CreatedBy":lit("Databricks")
                    }).whenNotMatchedInsert(values={
                        "tgt.AddressSK":"src.AddressSK",
                        "tgt.CustomerID":"src.CustomerID",
                        "tgt.City":"src.City",
                        "tgt.Country":"src.Country",
                        "tgt.Hashvalue":"src.src_hash",
                        "tgt.CreatedDate":current_timestamp(),
                        "tgt.CreatedBy":lit("Databricks"),
                        "tgt.UpdatedDate":lit("9999-12-31"),
                        "tgt.UpdatedBy":lit("N/A"),
                    }).execute()
                  )


DataFrame[num_affected_rows: bigint, num_updated_rows: bigint, num_deleted_rows: bigint, num_inserted_rows: bigint]

In [0]:
%sql
select * from address_scdtype1

AddressSK,CustomerID,City,Country,Hashvalue,CreatedDate,UpdatedDate,CreatedBy,UpdatedBy
1,101,New York,USA,403127107,2026-02-13T01:40:33.377532Z,9999-12-31T00:00:00Z,Databricks,N/A
2,102,Manchester,UK,2429636690,2026-02-13T01:40:33.377532Z,9999-12-31T00:00:00Z,Databricks,N/A
4,104,Berlin,Germany,2681540978,2026-02-13T01:40:33.377532Z,9999-12-31T00:00:00Z,Databricks,N/A
3,103,Kitchener,Germany,2448937089,2026-02-13T02:01:36.993573Z,2026-02-13T02:01:36.993573Z,Databricks,Databricks
5,105,Frankfurt,Germany,3153943322,2026-02-13T02:01:36.993573Z,9999-12-31T00:00:00Z,Databricks,N/A
